In [2]:
!pip install pycryptodome base58 ecdsa

import hashlib
import base58
from ecdsa import SigningKey, SECP256k1
from Crypto.Hash import RIPEMD160

# Key-to-Hash160
def privkey_to_hash160(k: int, compressed: bool = True) -> bytes:
    try:
        if not (1 <= k < SECP256k1.order):
            return None
        sk = SigningKey.from_secret_exponent(k, curve=SECP256k1)
        vk = sk.get_verifying_key()
        pubkey = (b'\x02' if vk.to_string()[32] % 2 == 0 else b'\x03') + vk.to_string()[:32] if compressed else b'\x04' + vk.to_string()
        sha = hashlib.sha256(pubkey).digest()
        ripemd = RIPEMD160.new()
        ripemd.update(sha)
        return ripemd.digest()
    except:
        return None

# Hamming Distance with Bit Positions
def hamming_distance_bits(hash1: bytes, hash2: bytes) -> tuple[int, list[int]]:
    distance = 0
    flips = []
    for i in range(len(hash1)):
        xor = hash1[i] ^ hash2[i]
        for j in range(8):
            if xor & (1 << j):
                bit_index = (i * 8) + (7 - j)
                flips.append(bit_index)
                distance += 1
    return distance, flips

# Decode Base58Check address to get hash160
def decode_address(address: str) -> bytes:
    decoded = base58.b58decode(address)
    version = decoded[0]
    hash160 = decoded[1:-4]
    checksum = decoded[-4:]
    # Verify checksum
    expected_checksum = hashlib.sha256(hashlib.sha256(decoded[:-4]).digest()).digest()[:4]
    if checksum != expected_checksum:
        raise ValueError("Checksum verification failed")
    print(f"Decoded Address: Version={hex(version)}, Hash160={hash160.hex()}, Checksum={checksum.hex()}")
    return hash160

# Verify Hamming distance
def verify_hamming(target_addr, key, compressed):
    target_hash160 = decode_address(target_addr)
    h160 = privkey_to_hash160(key, compressed)
    if not h160:
        return None, None, "Invalid private key"
    ham, flips = hamming_distance_bits(h160, target_hash160)
    return ham, flips

# Main test
if __name__ == "__main__":
    target_addr = "12ib7dApVFvg82TXKycWBNpN8kFyiAN1dr"
    key = 0x2b3c4d5e6f7890cf  # Best key from Test 31
    compressed = True
    ham, flips = verify_hamming(target_addr, key, compressed)
    print(f"Hamming Distance: {ham}")
    print(f"Bit positions to flip: {flips}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.6/150.6 kB 10.6 MB/s eta 0:00:00
Decoded Address: Version=0x0, Hash160=12d5a845f2b212ce0c3bd65a4035881d9219090e, Checksum=5a5067d1
Hamming Distance: 80
Bit positions to flip: [7, 6, 5, 4, 2, 1, 0, 15, 13, 10, 9, 8, 19, 17, 31, 30, 29, 25, 24, 39, 35, 34, 33, 32, 46, 44, 43, 41, 40, 51, 50, 48, 62, 61, 58, 57, 56, 69, 65, 64, 78, 77, 76, 75, 73, 72, 87, 86, 82, 81, 92, 91, 90, 89, 103, 102, 100, 96, 111, 110, 104, 117, 115, 113, 112, 127, 125, 124, 120, 134, 133, 132, 129, 128, 143, 140, 136, 151, 157, 155]


In [3]:
# Install necessary library
!pip install reportlab

from reportlab.lib.pagesizes import LETTER
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, PageBreak
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch

# Create PDF
doc = SimpleDocTemplate(
    "FinCEN_Investigative_Report.pdf",
    pagesize=LETTER,
    rightMargin=0.5*inch, leftMargin=0.5*inch,
    topMargin=0.75*inch, bottomMargin=0.75*inch
)

styles = getSampleStyleSheet()
# Define custom styles
styles.add(ParagraphStyle(name="DocTitle", fontSize=16, leading=20, alignment=1, spaceAfter=12))
styles.add(ParagraphStyle(name="DocSubTitle", fontSize=14, leading=18, alignment=1, spaceAfter=10))
styles.add(ParagraphStyle(name="InstructionsTitle", fontSize=12, leading=16, spaceAfter=8))
styles.add(ParagraphStyle(name="Body", fontSize=10, leading=14))

elements = []

# Title
elements.append(Paragraph("U.S. Department of the Treasury", styles["DocTitle"]))
elements.append(Paragraph("Financial Crimes Enforcement Network (FinCEN)", styles["DocSubTitle"]))

# Table data
data = [
    ["Entity", "Bank Name", "Account No / IBAN", "Routing / ABA", "SWIFT / BIC"],
    ["AG Financial Products Inc", "Bank of America (FTRA)", "PER_9846", "-", "-"],
    ["Romeo Jasper Grobbelaar", "Banco de Loja (Ecuador)", "2901249702", "-", "-"],
    ["Abdullah Mohammad A Saimair", "Banque Saudi Fransi", "SA811000001400018133319", "-", "BSFSARXX"],
    ["Anisa Hysa / CIS Luxury Hideaway RE GmbH", "Ziraat Bank A.S.", "TR03 0001 0008 0996 9752 6050 15", "-", "TCZBTR2AXXX"],
    ["AGP Global Capital", "Citibank (Wall St, NY)", "36204566", "021000089", "CITIUS33"],
    ["AGP Global Capital", "Canadian Imperial Bank of Commerce (CIBC)", "XXXXXX8282", "-", "CIBCCATTXXX"],
    ["LI & Partners Global", "Wells Fargo Bank", "...5562", "-", "-"],
    ["GMC Data LLC", "Relay Financial Technologies Inc", "200000681352", "064209588", "RFTECAT2"],
    ["Barbar Inc (IOLTA Trust)", "JPMorgan Chase Bank, N.A.", "882708610", "Dom: 322271627 / Intl: 021000021", "CHASUS33"],
    ["JPMorgan Clearing Corp (JPMC)", "JPMorgan Chase Bank, N.A.", "06601633", "021000021", "CHASUS33"],
    ["The Insurance Guys", "PNC Bank (Business Checking X0269)", "4731930269", "Dom: 043000096 / Intl: 071921891", "PNCCUS33"],
    ["Gray Lion Funding Solutions LLC", "U.S. Bank – Cornelius Branch", "169702306377", "123000220", "-"],
    ["XLIN Limited", "Bank of Communications (Hong Kong) Ltd", "382-554104970801", "-", "COMMHKHXXX"],
    ["Shropshire Savings & Loans Ltd", "Deutsche Bank AG", "0920009800 (IBAN DE74500700100920009800)", "-", "DEUTDEFF"],
    ["Kingdom Investment Corporation Ltd", "Deutsche Bank AG", "DE72500700100613341717", "-", "DEUTDEFFXXX"],
    ["GMC Data LLC", "Bluevine Inc", "875106734093", "125109019", "BLVNUS32"],
]

col_widths = [1.6*inch, 2.0*inch, 1.8*inch, 1.2*inch, 1.2*inch]

table = Table(data, colWidths=col_widths, repeatRows=1)
table.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor("#003366")),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('ALIGN', (0,0), (-1,-1), 'LEFT'),
    ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
    ('GRID', (0,0), (-1,-1), 0.5, colors.black),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTNAME', (0,1), (-1,-1), 'Helvetica'),
    ('FONTSIZE', (0,0), (-1,-1), 9),
    ('LINEBEFORE', (1,0), (1,-1), 0.5, colors.black),
    ('LINEAFTER', (3,0), (3,-1), 0.5, colors.black),
]))
elements.append(table)
elements.append(PageBreak())

# Instructions
elements.append(Paragraph("Investigative Instructions", styles["InstructionsTitle"]))
instructions = [
    "1. Retrieve all transaction records, bank statements, and SWIFT logs for each account listed.",
    "2. Map and chart fund flows across all correspondent and intermediary banks, including nested routing.",
    "3. Identify cryptocurrency off-ramps (CEX, DEX, wallets) and pull on-chain records for traceability.",
    "4. Obtain KYC/AML and CDD documentation for all counterparties, focusing on shell entities and UBOS.",
    "5. Trace all inflows and outflows to ultimate master fund or omnibus accounts, tagging circular movements.",
    "6. Screen for layering, structuring, and other laundering typologies, documenting each flagged pattern.",
    "7. Coordinate with blockchain analytics teams for wallet attribution and forensic linkage.",
    "8. Compile a Suspicious Activity Report (SAR) in compliance with FinCEN standards, attaching exhibits."
]
for line in instructions:
    elements.append(Paragraph(line, styles["Body"]))

# Build PDF
doc.build(elements)

print("✅ FinCEN_Investigative_Report.pdf generated successfully.")


✅ FinCEN_Investigative_Report.pdf generated successfully.
